In [442]:
import confnotebook

In [443]:
from pathlib import Path

source = Path("../examples/test/full/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 10
[1] 126164
[2] 14964427_Енисейская ТГК-13-БРАЗ
[3] 14976087_АвеларСолар Тех-БРАЗ-1
[4] 15120979_Форвард Энерго-БРАЗ-1
[5] 15235008_ОГК-2-БРАЗ-1
[6] 25
[7] 33
[8] 4
[9] 44
[10] 7-1
[11] АС ВНИИМ Менделеева Д.И. - БРАЗ на 31.12.24
[12] АС ВНИИМ Менделеева Д.И. - РУ на 31.12.24
[13] АС КРЕЗОЛ-САЗ на 31.08.25
[14] АС Охрана Металлург-САЗ на 31.12.25
[15] АС РУ- ВОСЬМОЙ ВЕТРОПАРК
[16] АС Фрейт Линк-БРАЗ на 30.09.25
[17] АСР СДД 2 кв.2024 (подп. к-а)
[18] Акт сверки взаимных расчетов №00000379931 от 30.04.2024
[19] Акт сверки №0000
[20] Акт сверки №MOW00-0087974   от 10.06.2024
[21] Акт сверки №ТРБП-000006 от 10.01.2024
[22] Браз-Юнигрин Пауэр
[23] ЕВР-НКАЗ
[24] Неформализованный_первичный_документ_23_ИИА_03_00342_от_31_03
[25] Неформализованный_первичный_документ_23_ИИА_03_02596_от_31_03
[26] Неформализованный_первичный_документ_23_ИИА_03_03623_от_31_03
[27] Неформализованный_первичный_документ_23_ИИА_06_01794_от_30_06
[28] ПР_АС КРЕЗОЛ-САЗ на 31.08.25
[29] ПР_АС Фрейт Линк-

In [444]:
IDX_FILE = 36

In [445]:
from vision_core.debug_image_observer import DebugImageObserver

file = files[IDX_FILE]
# output_dir = f"../examples/output/{file.stem}"

# debug_image_observer = DebugImageObserver(output_dir=output_dir)

In [446]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline = DocumentBuildPipeline()

document = pipeline.build(file.read_bytes())

Creating model: ('PP-OCRv5_server_det', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/PP-OCRv5_server_det')


Creating model: ('cyrillic_PP-OCRv5_mobile_rec', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/cyrillic_PP-OCRv5_mobile_rec')
2026-04-23 12:10:32.920 | INFO     | vision_core.pipelines.build_document:build:89 - Обработка страницы 0 с dpi 200...
2026-04-23 12:10:32.938 | INFO     | vision_core.pipelines.build_document:_process_page:146 - Предобработка изображения...
2026-04-23 12:10:32.941 | INFO     | vision_core.pipelines.build_document:_process_page:148 - Предобработка завершена.
2026-04-23 12:10:32.941 | INFO     | vision_core.pipelines.build_document:_process_page:150 - Коррекция ориентации и наклона...
2026-04-23 12:10:32.949 | DEBUG    | vision_core.preprocessor.image_orientation:process:41 - Ориентация страницы: 0° с точностью 0.9296
2026-04-23 12:10:32.958 | DEBUG    | vision_core.preprocessor.image_orientation:_correct_perspective_from_table:171 - Перспектива: [[113.5, 662.5], [1515.5, 662.5], [1515.5, 1802.0], [113.5, 1802.0]] -> [109,658] 1412x1149
2026-04-23 

In [447]:
def get_cell_covering(table, row: int, col: int):
    for cell in table.get_rows()[row]:
        if cell.col <= col < cell.col + cell.colspan:
            return cell
    return None


summary_text = ""
summary_cell_text: list[str] = []
for page in document.pages:
    text_paragraph = " ".join(paragraph.text for paragraph in page.paragraphs)
    for table in page.tables:
        if table.continuation_of is not None:
            continue
        dc = table.dc_cols
        num_row = table.get_dc_header_row()
        if num_row == -1 or not dc:
            continue
        seen_cells: set[int] = set()
        for i, col in enumerate(sorted(dc)):
            for j in range(num_row):
                cell = get_cell_covering(table, j, col)
                if cell is None or id(cell) in seen_cells:
                    continue
                seen_cells.add(id(cell))
                cell_text = cell.value.strip()
                if cell_text:
                    summary_cell_text.append(cell_text)

    summary_text += text_paragraph + " "

print("SUMMARY CELL TEXT:")
print(summary_cell_text)
print("\nSUMMARY TEXT:")
print(summary_text)

SUMMARY CELL TEXT:
['По данным Продавца', 'По данным Покупателя', 'По данным Продавца', 'По данным Покупателя']

SUMMARY TEXT:
012-40-904-25167860 АКТ СВЕРКИ РАСЧЕТОВ № 23/ИИА-03-01142 от 31 марта 2025 г. KOM-H-03.2025-01142 Между Акционерное общество "Интер РАО - Электрогенерация" и Акционерное общество "РУСАЛ Новокузнецкий Алюминиевый Завод" по Договору купли-продажи мощности по результатам конкурентного отбора мощности № KOM-30014452-INTRAOEG-NKUZALUM-24-VV-2 от 24.09.2019 за 1 квартал 2025 г. 31 марта 2025 г. (pyб.) От Продавца: АО "Интер РАО - Электрогенерация" От Покупателя: Акционерное общество "РУСАЛ Новокузнецкий Алюминиевый Завод" /Голубева Мария Александровна / на основании доверенности от 16.12.2024 № ДПП/2024/ЭГ/47 / 012-40-904-25167860 АКТ СВЕРКИ РАСЧЕТОВ № 23/ИИА-03-01142 от 31 марта 2025 г. KOM-H-03.2025-01142 Между Акционерное общество "Интер РАО - Электрогенерация" и Акционерное общество "РУСАЛ Новокузнецкий Алюминиевый Завод" по Договору купли-продажи мощности по рез

In [448]:
import re

_LAT2CYR = str.maketrans({
    'A':'А','B':'В','C':'С','E':'Е','H':'Н','K':'К','M':'М','O':'О','P':'Р','T':'Т','X':'Х','Y':'У',
    'a':'а','c':'с','e':'е','o':'о','p':'р','x':'х','y':'у','k':'к','m':'м','h':'н','b':'в','t':'т',
    'R':'Р','r':'р','V':'В','v':'в'
})
_QUOTES = '«»\u201c\u201d\u201e\u2018\u2019\u201a\u2039\u203a'
_QUOTE_NORM = str.maketrans(_QUOTES, '"' * len(_QUOTES))

def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = text.translate(_QUOTE_NORM)
    text = re.sub(r'"{2,}', '"', text)          # "" -> "
    text = re.sub(r'(\S)"', r'\1 "', text)      # ОБЩЕСТВО" -> ОБЩЕСТВО "
    text = text.translate(_LAT2CYR)
    text = text.upper().replace('Ё', 'Е')
    text = re.sub(r'\b000\b', 'ООО', text)
    text = re.sub(r'\s+', ' ', text, flags=re.UNICODE)
    return text.strip()



normalized_text = normalize_text(summary_text)

summary_cell_text_norm = [normalize_text(cell) for cell in summary_cell_text]


print("SUMMARY CELL TEXT:")
print(summary_cell_text_norm)
print("\nSUMMARY TEXT:")
print(normalized_text)

SUMMARY CELL TEXT:
['ПО ДАННЫМ ПРОДАВЦА', 'ПО ДАННЫМ ПОКУПАТЕЛЯ', 'ПО ДАННЫМ ПРОДАВЦА', 'ПО ДАННЫМ ПОКУПАТЕЛЯ']

SUMMARY TEXT:
012-40-904-25167860 АКТ СВЕРКИ РАСЧЕТОВ № 23/ИИА-03-01142 ОТ 31 МАРТА 2025 Г. КОМ-Н-03.2025-01142 МЕЖДУ АКЦИОНЕРНОЕ ОБЩЕСТВО "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ " И АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД " ПО ДОГОВОРУ КУПЛИ-ПРОДАЖИ МОЩНОСТИ ПО РЕЗУЛЬТАТАМ КОНКУРЕНТНОГО ОТБОРА МОЩНОСТИ № КОМ-30014452-INТРАОЕG-NКUZАLUМ-24-ВВ-2 ОТ 24.09.2019 ЗА 1 КВАРТАЛ 2025 Г. 31 МАРТА 2025 Г. (РУБ.) ОТ ПРОДАВЦА: АО "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ " ОТ ПОКУПАТЕЛЯ: АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД " /ГОЛУБЕВА МАРИЯ АЛЕКСАНДРОВНА / НА ОСНОВАНИИ ДОВЕРЕННОСТИ ОТ 16.12.2024 № ДПП/2024/ЭГ/47 / 012-40-904-25167860 АКТ СВЕРКИ РАСЧЕТОВ № 23/ИИА-03-01142 ОТ 31 МАРТА 2025 Г. КОМ-Н-03.2025-01142 МЕЖДУ АКЦИОНЕРНОЕ ОБЩЕСТВО "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ " И АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД " ПО ДОГОВОРУ КУПЛИ-ПРОДАЖИ МОЩНОСТИ 

In [449]:
ORGFORMS = {
    'АО':   'акционерное общество',
    'ОАО':  'открытое акционерное общество',
    'ЗАО':  'закрытое акционерное общество',
    'ООО':  'общество с ограниченной ответственностью',
    'ИП':   'индивидуальный предприниматель',
    'ПАО':  'публичное акционерное общество',
    'НП':   'некоммерческое партнерство',
    'ГУП':  'государственное унитарное предприятие',
    'МУП':  'муниципальное унитарное предприятие',
    'ФГУП': 'федеральное государственное унитарное предприятие',
}
ORGFORMS_FULL2SHORT = {v.upper(): k for k, v in ORGFORMS.items()}
def ocr_robust(s: str) -> str:
    """
    Заменяет букву "О" на паттерн, который может соответствовать как "О",
    так и "0", для повышения устойчивости к ошибкам OCR.
    """
    return s.replace('О', '[О0]').replace('о', '[о0]')

org_forms_pattern = "|".join(
    ocr_robust(k) for k in sorted(ORGFORMS.keys(), key=len, reverse=True)
) + r'|(?:' + "|".join(
    ocr_robust(v) for v in ORGFORMS.values()
) + r')'

In [450]:
def fix_rusal(name: str) -> str:
    """
    Корректирует специфические ошибки в написании названия "РУСАЛ"
    (например, "РУСАЛСАЯНОГОРСК" -> "РУСАЛ САЯНОГОРСК").
    """
    # Используем группу захвата для сохранения первой буквы после "РУСАЛ"
    return re.sub(r'РУСАЛ([А-ЯЁ])', r'РУСАЛ \1', name)

def extract_org_names(text: str) -> list[str]:
    """
    Извлекает уникальные полные названия организаций из заданного текста.

    Функция использует два прохода:
    1. Поиск по юридическим формам (якорь: АО, ООО и т.д.).
    2. Поиск названий, начинающихся с "РУСАЛ" без явной формы.

    Args:
        text: Текст, из которого необходимо извлечь названия организаций.

    Returns:
        list[str]: Список уникально найденных полных названий организаций.
    """
    results: list[str] = []
    seen_full: set[str] = set()
    seen_names: set[str] = set()

    # Регулярное выражение для поиска всех юридических форм
    org_re = re.compile(r'\b(?:' + org_forms_pattern + r')\b', re.IGNORECASE)

    # Проход 1: Поиск по юридическим формам (Якорь)
    for m in org_re.finditer(text):
        # Берем контекст 150 символов после найденной формы
        rest = text[m.end(): m.end() + 150]

        # Захватывает текст в кавычках, игнорируя вложенные
        q = re.search(r'"((?:[^"]*"(?=[А-ЯЁA-Za-zа-яё]))*[^"]*)"', rest)
        # ищем текст в кавычках, который может быть без пробелов
        if not q:
            q = re.search(r'"(\S+)', rest)

        if not q:
            continue

        name = fix_rusal(q.group(1).strip()).replace('"', '')
        matched = m.group().strip().replace('0', 'О').upper()
        short_form = ORGFORMS_FULL2SHORT.get(matched, matched)  # полная -> краткая, или уже краткая
        full_name = name + ', ' + short_form

        if full_name not in seen_full:
            seen_full.add(full_name)
            seen_names.add(name)
            results.append(full_name)

    # Проход 2: Поиск "РУСАЛ..." без орг.формы
    # Ищем любые кавычки, начинающиеся с "РУСАЛ", независимо от того, что следует за ними.
    # Паттерн r'"(РУСАЛ[^"]*)"' захватывает все, что в кавычках и начинается с РУСАЛ.
    for q in re.finditer(r'"(РУСАЛ[^"]*)"', text):
        name = fix_rusal(q.group(1).strip()).replace('"', '')
        if name not in seen_names:
            seen_names.add(name)
            results.append(name + ", ")

    return results


print("Из ячеек:", extract_org_names(" ".join(summary_cell_text_norm)))
print("\nИз полного текста:", extract_org_names(normalized_text))


Из ячеек: []

Из полного текста: ['ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО', 'РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО', 'ПФ СКБ КОНТУР, АО', 'ИНТЕР ПОДПИСИ 8 ПОДПИСЬ СООТВЕТСТВУЕТ ФАЙЛУ Е93В970601D5 4С85685 РАО-ЭЛЕКТРОГЕНЕРАЦИЯ, АО']


In [ ]:
from enum import Enum


class Role(Enum):
    BUYER = 0
    SELLER = 1
    UNKNOWN = -1


ROLE_GLOSSARY = {
    r'\bОТ ПОКУПАТЕЛЯ\b': 'BUYER_SOURCE',
    r'\bОТ ПРОДАВЦА\b':   'SELLER_SOURCE',
    r'\bМЕЖДУ\b':         'PARTICIPATION_SCOPE',
}


def _org_token(full_name: str) -> str:
    return full_name.split(",", 1)[0].strip()


def _find_events(text: str) -> list[dict]:
    found = []
    for pattern, role in ROLE_GLOSSARY.items():
        for m in re.finditer(pattern, text, re.IGNORECASE):
            found.append({"keyword": m.group(), "role": role,
                          "start_index": m.start(), "end_index": m.end()})
    return sorted(found, key=lambda x: x["start_index"])


def _find_orgs_in_span(text: str, left: int, right: int, orgs: list[str]) -> list[str]:
    span = text[left:right]
    return [o for o in orgs if _org_token(o) and _org_token(o) in span]


def _find_working_pair(text: str, orgs: list[str], events: list[dict]) -> list[str]:
    """Два контрагента рядом с якорем МЕЖДУ."""
    anchor = next((e for e in events if e["role"] == "PARTICIPATION_SCOPE"), None)
    if anchor:
        window = text[anchor["end_index"]: anchor["end_index"] + 400]
        hits = sorted(
            [(window.find(_org_token(o)), o) for o in orgs if _org_token(o) in window]
        )
        pair = [o for _, o in hits[:2]]
        if len(pair) == 2:
            return pair
    return orgs[:2]


def _apply_symmetry(roles: dict) -> None:
    buyers   = [o for o, r in roles.items() if r == Role.BUYER]
    sellers  = [o for o, r in roles.items() if r == Role.SELLER]
    unknowns = [o for o, r in roles.items() if r == Role.UNKNOWN]
    if buyers and unknowns and not sellers:
        for o in unknowns:
            print(f"Правило симметрии: {o} -> SELLER")
            roles[o] = Role.SELLER
    elif sellers and unknowns and not buyers:
        for o in unknowns:
            print(f"Правило симметрии: {o} -> BUYER")
            roles[o] = Role.BUYER

def _deduplicate_orgs(orgs: list[str]) -> list[str]:
    tokens = [_org_token(o) for o in orgs]
    return [
        org for i, org in enumerate(orgs)
        if not any(tokens[i] in tokens[j] and tokens[i] != tokens[j] for j in range(len(tokens)))
    ]

def assign_roles(text: str, orgs: list[str]) -> dict[str, Role]:
    events = _find_events(text)
    print("Найдены события:", events)
    pair   = _find_working_pair(text, orgs, events)
    print("Рабочая пара:", pair)
    roles  = {o: Role.UNKNOWN for o in pair}

    # 1) РУСАЛ -> всегда BUYER (жёсткое правило)
    for o in pair:
        if "РУСАЛ" in _org_token(o):
            roles[o] = Role.BUYER
            print(f"Правило: {o} содержит 'РУСАЛ' -> BUYER")
    _apply_symmetry(roles)

    # 2) Явные якоря ОТ ПОКУПАТЕЛЯ / ОТ ПРОДАВЦА (только для UNKNOWN)
    for idx, event in enumerate(events):
        if event["role"] not in {"BUYER_SOURCE", "SELLER_SOURCE"}:
            continue
        next_start = events[idx + 1]["start_index"] if idx + 1 < len(events) else len(text)
        target = Role.BUYER if event["role"] == "BUYER_SOURCE" else Role.SELLER
        for o in _find_orgs_in_span(text, event["end_index"], next_start, pair):
            if roles[o] == Role.UNKNOWN:
                roles[o] = target
                print(f"Правило: {o} находится в диапазоне {event['keyword']} -> {target.name}")
    _apply_symmetry(roles)

    # 3) Позиционный фоллбек: первый -> SELLER, второй -> BUYER
    unknowns = [o for o, r in roles.items() if r == Role.UNKNOWN]
    if unknowns:
        roles[unknowns[0]] = Role.SELLER
        print(f"Правило позиционного фоллбека: {unknowns[0]} -> SELLER")
        for o in unknowns[1:]:
            roles[o] = Role.BUYER
            print(f"Правило позиционного фоллбека: {o} -> BUYER")

    return roles


# --- запуск ---
orgs_name = extract_org_names(" ".join(summary_cell_text_norm))
orgs_name_full = extract_org_names(normalized_text)

if len(orgs_name) >= 2:
    orgs_name_full_filtered = orgs_name
elif orgs_name:
    orgs_name_set = set(orgs_name)
    orgs_name_full_filtered = [
        o for o in orgs_name_full if any(n in o for n in orgs_name_set)
    ]
else:
    orgs_name_full_filtered = orgs_name_full

orgs_name_full_filtered = _deduplicate_orgs(orgs_name_full_filtered)
print("Организации (после дедупликации):", orgs_name_full_filtered)


roles_by_org = assign_roles(normalized_text, orgs_name_full_filtered)

for org, role in roles_by_org.items():
    print(f"- {org} -> {role.name}")

buyers  = [o for o, r in roles_by_org.items() if r == Role.BUYER]
sellers = [o for o, r in roles_by_org.items() if r == Role.SELLER]
print("\nBUYER:", buyers)
print("SELLER:", sellers)


Организации (после дедупликации): ['ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО', 'РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО', 'ПФ СКБ КОНТУР, АО', 'ИНТЕР ПОДПИСИ 8 ПОДПИСЬ СООТВЕТСТВУЕТ ФАЙЛУ Е93В970601D5 4С85685 РАО-ЭЛЕКТРОГЕНЕРАЦИЯ, АО']
Найдены события: [{'keyword': 'МЕЖДУ', 'role': 'PARTICIPATION_SCOPE', 'start_index': 98, 'end_index': 103}, {'keyword': 'ОТ ПРОДАВЦА', 'role': 'SELLER_SOURCE', 'start_index': 401, 'end_index': 412}, {'keyword': 'ОТ ПОКУПАТЕЛЯ', 'role': 'BUYER_SOURCE', 'start_index': 449, 'end_index': 462}, {'keyword': 'МЕЖДУ', 'role': 'PARTICIPATION_SCOPE', 'start_index': 715, 'end_index': 720}, {'keyword': 'ОТ ПРОДАВЦА', 'role': 'SELLER_SOURCE', 'start_index': 1018, 'end_index': 1029}, {'keyword': 'ОТ ПОКУПАТЕЛЯ', 'role': 'BUYER_SOURCE', 'start_index': 1066, 'end_index': 1079}]
Рабочая пара: ['ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО', 'РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО']
Правило: РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО содержит 'РУСАЛ' -> BUYER
Правило симметрии: ИНТЕР РАО